# Static Connection Headers (`[source.headers]`) — REST table-gateway

> **Design spec — integration-focused.** A general primitive: attach **constant, templated headers** to every request a REST connection makes, *alongside* the existing OAuth2/auth header. **Google Ads is consumer #1** (it requires a `developer-token` header in addition to the OAuth2 Bearer), but the feature is generic — any API that needs a fixed header (API version, tenant id, `developer-token`) uses the same mechanism.

| | |
|---|---|
| **Status** | Design / approved approach, pending spec review |
| **Design epic** | `bd-1irz` |
| **Intent** | C — general static-headers primitive; Google Ads as the worked example |
| **Approach** | ① source-level templated `[source.headers]` (no enum change → zero `AuthCfg` E0004 risk) |
| **Crate** | `crates/spur-notebook/rest-table-gateway` |
| **Author** | brain · **Date** 2026-06-03 |

## The one-sentence problem

Today the gateway can send **exactly one** auth header per request (`apply_auth` → `ResolvedAuth::{Bearer|Header|Basic|QueryParam}`) plus an optional idempotency header. Google Ads — and many enterprise APIs — need a **second, constant header** (`developer-token`) on *every* call **in addition to** the OAuth2 Bearer. There is no mechanism for that. This spec adds connection-level static headers.

## What the empirical self-check established (epic `bd-1irz`)

Everything else Google Ads needs **already works and is tested** — confirmed against passing tests + Google's REST docs:
- ✅ **POST a GAQL body → rows**: `tests::action_post_renders_typed_columns` (POST, `in="body"` arg, `columns` → `act()` returns rows).
- ✅ **customer-id in path**: Path-arg substitution (`/orders/{token_id}` → `/orders/tok1`).
- ✅ **OAuth2 Bearer**: Approach B (`resolve_auth` → `oauth::access_token`), shipped.
- ✅ **Queryable as a table function**: `ApiActionVTab`.

**The only true gap is the second static header.** This spec closes exactly that.

## Scope

**In:** `[source.headers]` parse + templated resolve + apply on **both** the read (`fetch_rows`) and action (`send_request`) paths; a Google Ads GAQL worked-example manifest; a wiremock E2E proving Bearer **and** `developer-token` are sent together and rows come back.

**Out (separate follow-ups):** read-vs-write gating cleanup (a GAQL *read* currently rides the `allow_writes` action path); GAQL `nextPageToken` pagination."

## 1 · Integration map — where static headers plug in

The feature follows the **exact same shape as the shipped auth path**: *resolve in the adapter, apply in http*. Green = exists today. Amber = new. The new code is purely additive — **no enum is touched**, so the `AuthCfg` E0004 exhaustiveness trap cannot fire.

```mermaid
flowchart TB
    subgraph MANIFEST["📦 MANIFEST"]
        SRC["SourceCfg<br/><i>manifest.rs:23</i>"]
        HDRS["NEW: headers: IndexMap&lt;String,String&gt;<br/>values: ${connectionConfig.*}"]
        AUTH["auth: AuthCfg<br/>(unchanged)"]
    end

    subgraph ADAPTER["⚙️ ManifestAdapter  (resolve)"]
        SCAN["scan()  — read path"]
        ACT["act()  — action path"]
        RAUTH["resolve_auth → ResolvedAuth<br/><i>(shipped)</i>"]
        RHDRS["NEW: resolve_headers(&amp;ctx)<br/>template each value via<br/>resolve_template + ConnectionContext<br/>→ Vec&lt;(String,String)&gt;"]
    end

    subgraph HTTP["🌐 http.rs  (apply)"]
        FETCH["fetch_rows(HttpFetch)<br/>+ NEW headers field"]
        SEND["send_request(HttpAction)<br/>+ NEW headers field"]
        APPLYAUTH["apply_auth(req, &amp;auth)<br/><i>(shipped — single header)</i>"]
        APPLYHDRS["NEW: for (k,v) in headers<br/>req.header(k, v)"]
        OUT(["outbound request<br/>Authorization: Bearer …<br/>+ developer-token: …"])
    end

    SRC --> HDRS
    SRC --> AUTH
    SCAN --> RAUTH
    SCAN --> RHDRS
    ACT --> RAUTH
    ACT --> RHDRS
    HDRS -. "templated by" .-> RHDRS
    AUTH -. "resolved by" .-> RAUTH
    RAUTH == "ResolvedAuth" ==> FETCH
    RAUTH == "ResolvedAuth" ==> SEND
    RHDRS == "Vec&lt;(k,v)&gt;" ==> FETCH
    RHDRS == "Vec&lt;(k,v)&gt;" ==> SEND
    FETCH --> APPLYAUTH
    SEND --> APPLYAUTH
    APPLYAUTH --> APPLYHDRS
    APPLYHDRS --> OUT

    classDef shipped fill:#1f3a2e,stroke:#3fb27f,color:#cdebd9;
    classDef new fill:#3a2f1f,stroke:#d9a441,color:#f0e2c4;
    classDef io fill:#2a2240,stroke:#9a7fd9,color:#e2d9f5;
    class SRC,AUTH,SCAN,ACT,RAUTH,FETCH,SEND,APPLYAUTH shipped;
    class HDRS,RHDRS,APPLYHDRS new;
    class OUT io;
```

**Reading the diagram.** The amber nodes are the entire change: a manifest field, a `resolve_headers` sibling to `resolve_auth`, a `headers` field threaded onto the two request structs, and a 2-line apply loop that runs **after** `apply_auth`. The `==>` edges show static headers and resolved auth travelling the *same* path to the *same* two request builders — read (`fetch_rows`) and action (`send_request`) get the feature uniformly."

## 2 · Request-build sequence — Bearer + developer-token together

The headline guarantee: a single request carries **both** the OAuth2 Bearer (from `resolve_auth`) **and** the constant `developer-token` (from `resolve_headers`). Order matters — static headers apply *after* auth, and `authorization` is forbidden as a static-header name so it can never shadow the Bearer.

```mermaid
sequenceDiagram
    autonumber
    participant Q as DuckDB query<br/>SELECT … FROM google_ads_search(…)
    participant A as ManifestAdapter::act
    participant RA as resolve_auth
    participant RH as resolve_headers
    participant OAuth as oauth::access_token
    participant S as send_request (http.rs)
    participant G as Google Ads API

    Q->>A: ActionRequest{ customer_id, query }
    A->>RA: resolve_auth()
    RA->>OAuth: Oauth2Refresh grant (cached, auto-refresh)
    OAuth-->>RA: access_token
    RA-->>A: ResolvedAuth::Bearer(token)
    A->>RH: resolve_headers(&ctx)
    Note over RH: template "${connectionConfig.developer_token}"<br/>← SPUR_CONN_DEVELOPER_TOKEN
    RH-->>A: [("developer-token","<dev-tok>")]
    A->>S: HttpAction{ method:POST, url, body:{query}, auth, headers }
    S->>S: apply_auth(req, auth)      → Authorization: Bearer …
    S->>S: for (k,v) in headers       → developer-token: <dev-tok>
    S->>G: POST /customers/{id}/googleAds:search<br/>Authorization: Bearer …<br/>developer-token: …<br/>{ "query": "SELECT …" }
    G-->>S: 200 { "results": [ … ] }
    S-->>A: (200, body)
    A->>A: columns + response_path → rows_to_batch
    A-->>Q: Vec<RecordBatch>  (rows)
```

**Why this ordering is safe**
- **Auth first, headers second** (steps 14→15): a static header can *add* to the request but the Bearer is already set; we additionally **reject `authorization` as a static-header key** at parse/resolve time so a misconfigured manifest can't append a second, conflicting `Authorization`.
- **Secret never in the manifest** (step 9): `developer-token`'s value is `${connectionConfig.developer_token}`, resolved from `SPUR_CONN_DEVELOPER_TOKEN` at request time — same mechanism the gateway already uses for templated base URLs.
- **Auth stays auth** — `resolve_headers` is a *sibling* of `resolve_auth`, not a change to it. The `AuthCfg` enum and its 3 exhaustive match sites are untouched."

## 3 · Module DAG — new vs. changed

Five small touch-points, all additive. The widest blast-radius is two structs gaining a field.

```mermaid
flowchart LR
    subgraph existing["EXISTING — extend in place"]
        SRC["SourceCfg<br/><i>manifest.rs:23</i>"]
        MA["ManifestAdapter::scan / act<br/><i>manifest_adapter.rs</i>"]
        RAUTH["resolve_auth<br/><i>(pattern to mirror)</i>"]
        HF["HttpFetch<br/><i>http.rs</i>"]
        HA["HttpAction<br/><i>http.rs:27</i>"]
        FR["fetch_rows<br/><i>http.rs</i>"]
        SR["send_request<br/><i>http.rs:38</i>"]
    end

    subgraph new["NEW — add"]
        F1["+ headers: IndexMap field"]
        F2["resolve_headers(&amp;ctx)<br/>+ reject 'authorization' key"]
        F3["+ headers: Vec<(String,String)> field"]
        F4["+ apply loop after apply_auth"]
        EX["Google Ads manifest<br/>+ GAQL wiremock E2E"]
    end

    SRC -->|gains| F1
    RAUTH -.->|sibling| F2
    MA -->|calls| F2
    HF -->|gains| F3
    HA -->|gains| F3
    FR -->|gains| F4
    SR -->|gains| F4
    F2 ==>|threads Vec| F3
    F4 ==>|verified by| EX

    classDef chg fill:#1d2b3a,stroke:#4f8fd9,color:#cfe3f7;
    classDef add fill:#3a2f1f,stroke:#d9a441,color:#f0e2c4;
    class SRC,MA,RAUTH,HF,HA,FR,SR chg;
    class F1,F2,F3,F4,EX add;
```

| # | File | Change | Risk |
|---|------|--------|------|
| 1 | `manifest.rs` `SourceCfg:23` | `#[serde(default)] headers: IndexMap<String,String>` | Low — additive serde |
| 2 | `manifest_adapter.rs` | `resolve_headers(&ctx)` — template values; reject `authorization` | Low — mirrors `resolve_auth` |
| 3 | `http.rs` `HttpFetch` / `HttpAction:27` | `+ headers: Vec<(String,String)>` | Low — additive field, 2 call sites each |
| 4 | `http.rs` `fetch_rows` / `send_request:38` | apply loop after `apply_auth` | Low — 2 lines each |
| 5 | tests + example | Google Ads manifest + GAQL wiremock E2E | Low — test-only |

**No enum, no trait, no public-API break.** The `AuthCfg` 3-site exhaustiveness constraint (the E0004 trap from prior work) is deliberately avoided by making headers a *separate* concept from auth."

## 4 · The Google Ads worked example (end-to-end)

The manifest that the feature unlocks — every line below maps to a capability proven in the empirical self-check, with the **one new** line highlighted.

```toml
[source]
name = "google_ads"
base_url = "https://googleads.googleapis.com/v17"
allow_writes = true                          # GAQL read rides the action path (gating cleanup = follow-up)
connection_config = ["developer_token"]      # → SPUR_CONN_DEVELOPER_TOKEN
auth = { scheme = "oauth2_refresh",
         token_url = "https://oauth2.googleapis.com/token",
         client_id_env = "GOOGLE_ADS_CLIENT_ID",
         client_secret_env = "GOOGLE_ADS_CLIENT_SECRET",
         refresh_token_env = "GOOGLE_ADS_REFRESH_TOKEN" }

[source.headers]                             # ★ THE ONE NEW THING
developer-token = "${connectionConfig.developer_token}"

[[action]]
name = "google_ads_search"
method = "POST"
path = "/customers/{customer_id}/googleAds:search"
response_path = "$.results"
[action.args]
customer_id = { in = "path", type = "Utf8", required = true }
query       = { in = "body", type = "Utf8", required = true }
[action.columns]
campaign_id = { json = "$.campaign.id",          type = "Utf8" }
impressions = { json = "$.metrics.impressions",  type = "Int64" }
clicks      = { json = "$.metrics.clicks",       type = "Int64" }
```

```mermaid
flowchart LR
    SQL["SELECT campaign_id, impressions<br/>FROM google_ads_search(<br/>customer_id := '1234567890',<br/>query := 'SELECT campaign.id,<br/>metrics.impressions FROM campaign')"]
    VTAB["ApiActionVTab<br/>(table function)"]
    REQ["POST /customers/1234567890/googleAds:search<br/>Authorization: Bearer ya29.…<br/>★ developer-token: …<br/>{ &quot;query&quot;: &quot;SELECT …&quot; }"]
    RESP["200 { results: [ {campaign,metrics}, … ] }"]
    ROWS[("Arrow rows<br/>campaign_id | impressions | clicks")]

    SQL --> VTAB --> REQ --> RESP --> ROWS
    classDef a fill:#1f3a2e,stroke:#3fb27f,color:#cdebd9;
    classDef n fill:#3a2f1f,stroke:#d9a441,color:#f0e2c4;
    class SQL,VTAB,RESP,ROWS a;
    class REQ n;
```

The **★ developer-token header** is the only thing that doesn't work today — everything else in this path is already shipped and tested."

## 5 · UX — making "a second secret header" legible

**Surface:** the REST connection panel + the query result, inside the notebook host shell.
**Direction:** *dark dev-tool* — same palette as the OAuth specs, mono numerics, low-chroma surfaces, one amber accent on the *new* concept (the static header). This reads as a continuation of the connection story, not a new product.

### The UX problem
A static header like `developer-token` is **invisible plumbing** — but it's also a **secret** and the #1 reason a Google Ads connection silently 401s. The UX job is to make the header (a) visible as part of the connection, (b) obviously *secret* (value never shown, sourced from env), and (c) *provably sent* — so when a query works, the user can see both `Authorization` and `developer-token` went out, and when it fails, they see *which* header was missing.

### What the user sees (three moments)
1. **Connection config** — the connection card lists a **Headers** section beside Auth: `developer-token → from developer_token`, badged **secret · env**. The value is masked (`••••`); the manifest only ever holds the `${connectionConfig.developer_token}` reference, never the token.
2. **Request inspector** — on first run, a collapsible "what we sent" shows the outbound header set with **both** `Authorization: Bearer ••••` *and* `developer-token: ••••` present — the concrete proof that the two-header requirement is satisfied. This is the anti-401 affordance.
3. **Result** — the GAQL `SELECT` returns a real table (`campaign_id | impressions | clicks`), so the payoff is a queryable dataset, not just "connected."

### Anti-slop guards
- **No fake green checkmark** — "connected" is shown by the **request inspector with real header names**, not a content-free success badge.
- **Secrets are masked, never invented** — values render as `••••` with a "from env" provenance, never a fake token string.
- **The error state names the missing header** — `developer-token absent → set SPUR_CONN_DEVELOPER_TOKEN`, not "something went wrong." A missing-header failure is the single most common Google Ads setup error, so it gets a first-class, specific message.

The cell below renders the interactive mock (`text/html`)."

In [2]:
# open-design artifact — static-header connection + request inspector (Google Ads)
from IPython.display import HTML

html = """
<!DOCTYPE html><html><head><meta charset="utf-8"><style>
  :root{
    --bg:#0e1116; --panel:#161b22; --panel2:#1b212b; --line:#2a323d;
    --txt:#c9d4e0; --muted:#7d8a99; --mono:'SF Mono',ui-monospace,Menlo,monospace;
    --grn:#3fb27f; --grn-bg:#16271f; --amb:#d9a441; --amb-bg:#241d10;
    --blu:#4f8fd9; --pur:#9a7fd9; --dgr:#e0556b; --dgr-bg:#2a1419;
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--bg);font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
       color:var(--txt);padding:26px;display:flex;justify-content:center}
  .wrap{width:680px}
  .switch{display:flex;gap:6px;margin-bottom:16px}
  .switch button{flex:1;background:var(--panel);border:1px solid var(--line);color:var(--muted);
       font-size:11px;letter-spacing:.04em;text-transform:uppercase;padding:7px 0;border-radius:6px;cursor:pointer}
  .switch button.on{color:var(--txt);border-color:#3d4b5c;background:var(--panel2)}
  .card{background:var(--panel);border:1px solid var(--line);border-radius:12px;overflow:hidden;
        box-shadow:0 12px 40px rgba(0,0,0,.45)}
  .hd{display:flex;align-items:center;gap:10px;padding:15px 18px;border-bottom:1px solid var(--line)}
  .hd .t{font-size:13px;font-weight:600}
  .hd .sub{font-size:11px;color:var(--muted)}
  .chip{margin-left:auto;display:flex;align-items:center;gap:7px;background:var(--panel2);
        border:1px solid var(--line);border-radius:20px;padding:4px 11px 4px 5px;font-size:12px}
  .chip .dot{width:20px;height:20px;border-radius:5px;background:#4285F4;color:#fff;display:grid;
        place-items:center;font-weight:800;font-size:12px}
  .body{padding:16px 18px 20px}
  .label{font-size:10px;letter-spacing:.09em;text-transform:uppercase;color:var(--muted);margin:0 0 8px}
  .sec{margin-top:18px}
  .sec:first-child{margin-top:0}
  .kv{display:flex;align-items:center;gap:10px;background:#0c1117;border:1px solid var(--line);
      border-radius:7px;padding:9px 11px;margin-top:7px;font-family:var(--mono);font-size:12.5px}
  .kv .k{color:#a9c7e8}
  .kv .arrow{color:var(--muted)}
  .kv .v{color:#cbd6e2}
  .kv .badge{margin-left:auto;font-family:-apple-system,sans-serif;font-size:10px;letter-spacing:.03em;
        text-transform:uppercase;border-radius:5px;padding:3px 8px;border:1px solid}
  .b-secret{color:var(--amb);border-color:#5c4a1f;background:var(--amb-bg)}
  .b-auth{color:var(--grn);border-color:#265c45;background:var(--grn-bg)}
  .mask{letter-spacing:1.5px;color:var(--muted)}
  .new{color:var(--amb)}
  .insp{background:#0c1117;border:1px solid var(--line);border-radius:8px;margin-top:8px;overflow:hidden}
  .insp .row{display:flex;gap:8px;padding:9px 12px;font-family:var(--mono);font-size:12px;border-top:1px solid #1a212b}
  .insp .row:first-child{border-top:none}
  .insp .hk{color:#7fb0e8;min-width:130px}
  .insp .hv{color:#cbd6e2}
  .ok{color:var(--grn)}
  .sqlbar{font-family:var(--mono);font-size:12px;background:#0c1117;border:1px solid var(--line);
          border-radius:7px;padding:10px 12px;color:#bcd;line-height:1.5;margin-top:7px}
  .sqlbar .kw{color:#c98fd9}
  table{width:100%;border-collapse:collapse;margin-top:8px;font-family:var(--mono);font-size:12.5px}
  th{text-align:left;color:var(--muted);font-weight:500;font-size:10.5px;letter-spacing:.05em;
     text-transform:uppercase;padding:6px 10px;border-bottom:1px solid var(--line)}
  td{padding:7px 10px;border-bottom:1px solid #161d27;color:#cbd6e2}
  td.num{text-align:right;color:#9fe0c4}
  .banner{border-radius:9px;padding:12px 14px;font-size:13px;line-height:1.5;display:flex;gap:10px;align-items:flex-start;margin-top:8px}
  .b-dgr{background:var(--dgr-bg);border:1px solid #5c2630;color:#f3c0c8}
  .hide{display:none}
  .foot{padding:10px 18px;border-top:1px solid var(--line);font-size:11px;color:var(--muted);
        display:flex;justify-content:space-between;font-family:var(--mono)}
</style></head><body><div class="wrap">

  <div class="switch">
    <button class="on" onclick="show('ok',this)">Connected · query</button>
    <button onclick="show('err',this)">Missing developer-token</button>
  </div>

  <div class="card">
    <div class="hd">
      <div><div class="t">Google Ads</div><div class="sub">REST connection · oauth2_refresh + static header</div></div>
      <div class="chip"><span class="dot">A</span>google_ads</div>
    </div>
    <div class="body">

      <!-- CONNECTED -->
      <div id="ok">
        <div class="sec">
          <div class="label">Credentials</div>
          <div class="kv"><span class="k">Authorization</span><span class="arrow">&larr;</span>
            <span class="v">OAuth2 refresh grant</span><span class="badge b-auth">auto-refresh</span></div>
          <div class="kv"><span class="k new">developer-token</span><span class="arrow">&larr;</span>
            <span class="v">${connectionConfig.developer_token}</span><span class="badge b-secret">secret · env</span></div>
        </div>

        <div class="sec">
          <div class="label">Request inspector &mdash; what went out</div>
          <div class="insp">
            <div class="row"><span class="hk">POST</span><span class="hv">/customers/1234567890/googleAds:search</span></div>
            <div class="row"><span class="hk">Authorization</span><span class="hv">Bearer <span class="mask">&bull;&bull;&bull;&bull;&bull;&bull;</span> <span class="ok">&check;</span></span></div>
            <div class="row"><span class="hk">developer-token</span><span class="hv"><span class="mask">&bull;&bull;&bull;&bull;&bull;&bull;</span> <span class="ok">&check;</span></span></div>
            <div class="row"><span class="hk">content-type</span><span class="hv">application/json</span></div>
          </div>
        </div>

        <div class="sec">
          <div class="label">Query</div>
          <div class="sqlbar"><span class="kw">SELECT</span> campaign_id, impressions, clicks<br>
            <span class="kw">FROM</span> google_ads_search(customer_id := '1234567890',<br>
            &nbsp;&nbsp;query := 'SELECT campaign.id, metrics.impressions, metrics.clicks FROM campaign')</div>
          <table>
            <thead><tr><th>campaign_id</th><th>impressions</th><th>clicks</th></tr></thead>
            <tbody>
              <tr><td>9 482 117 003</td><td class="num">128 940</td><td class="num">4 213</td></tr>
              <tr><td>9 482 117 884</td><td class="num">86 552</td><td class="num">2 901</td></tr>
              <tr><td>9 482 120 511</td><td class="num">41 209</td><td class="num">1 077</td></tr>
            </tbody>
          </table>
        </div>
      </div>

      <!-- ERROR -->
      <div id="err" class="hide">
        <div class="sec">
          <div class="label">Credentials</div>
          <div class="kv"><span class="k">Authorization</span><span class="arrow">&larr;</span>
            <span class="v">OAuth2 refresh grant</span><span class="badge b-auth">auto-refresh</span></div>
          <div class="kv"><span class="k new">developer-token</span><span class="arrow">&larr;</span>
            <span class="v">${connectionConfig.developer_token}</span><span class="badge b-secret">unset</span></div>
        </div>
        <div class="banner b-dgr"><span style="font-size:15px">&#9888;</span>
          <div><b>Missing header: developer-token.</b> The OAuth2 Bearer resolved, but
          <span style="font-family:var(--mono)">${connectionConfig.developer_token}</span> has no value &mdash;
          Google Ads rejects the request before any rows. <b>Set</b>
          <span style="font-family:var(--mono)">SPUR_CONN_DEVELOPER_TOKEN</span> and re-run. Nothing was sent with a half-built header set.</div></div>
      </div>

    </div>
    <div class="foot"><span>[source.headers]</span><span>resolve_headers &rarr; apply after apply_auth</span></div>
  </div>
</div>
<script>
  function show(id,btn){
    ['ok','err'].forEach(function(x){document.getElementById(x).classList.add('hide')});
    document.getElementById(id).classList.remove('hide');
    document.querySelectorAll('.switch button').forEach(function(b){b.classList.remove('on')});
    if(btn) btn.classList.add('on');
  }
</script>
</body></html>
"""
HTML(html)

campaign_id,impressions,clicks
9 482 117 003,128 940,4 213
9 482 117 884,86 552,2 901
9 482 120 511,41 209,1 077


## 6 · Integration contract (the seam guarantees)
1. **Resolve in adapter, apply in http** — `resolve_headers` lives beside `resolve_auth`; `http.rs` only *applies* an already-resolved `Vec<(String,String)>`. No templating logic leaks into `http.rs`.
2. **Uniform across read & action** — the same resolved headers thread into both `HttpFetch` (read tables) and `HttpAction` (POST/GAQL actions). One feature, both paths.
3. **Auth is untouched** — `AuthCfg`, `ResolvedAuth`, and the 3 exhaustive match sites (`resolve_auth`, `auth_to_toml`, `required_env_vars_from_manifest`) do not change. **No E0004 risk.**
4. **Secrets stay out of the manifest** — header values are `${connectionConfig.*}` references resolved from `SPUR_CONN_*` env at request time.
5. **`authorization` is a reserved key** — rejected at parse/resolve so a static header can never append a second, conflicting `Authorization` (since `req.header()` appends rather than replaces).
6. **Empty by default** — `#[serde(default)]`; every existing manifest is byte-for-byte unaffected.

## 7 · Security model
- Header values are masked in any UI/inspector (`••••`), never rendered.
- The developer-token never appears in the manifest, logs, or error messages — only its env-var *name* is surfaced on failure.
- Reserved-key guard prevents auth shadowing.

## 8 · Acceptance criteria
- [ ] `[source.headers]` parses into `SourceCfg.headers` (new test); manifests without it still parse (default empty).
- [ ] `resolve_headers` templates `${connectionConfig.*}` → values; a missing env var yields the same "needs credential" error path as templated base URLs.
- [ ] Declaring `authorization` (case-insensitive) as a static header is rejected with a clear error.
- [ ] A **read** table with a static header sends it on the GET (wiremock asserts header present).
- [ ] An **action** with OAuth2 Bearer **+** a static `developer-token` sends **both** headers on the POST (wiremock asserts both), with a `{query}` body, and extracts rows from `results[]`.
- [ ] `cargo test -p spur-rest-table-gateway` and `cargo check -p spur-notebook` both green.

## 9 · Task decomposition (for `writing-plans`)
- **T1** — `SourceCfg.headers` parse + `resolve_headers(&ctx)` helper + `authorization`-reserved guard + unit tests (`manifest.rs`, `manifest_adapter.rs`). Root.
- **T2** — `headers: Vec<(String,String)>` on `HttpFetch`/`HttpAction`; apply loop after `apply_auth` in `fetch_rows`/`send_request`; wire `scan`/`act`; wiremock tests (read sends header; action sends Bearer+developer-token). Depends on T1.
- **T3** — Google Ads GAQL worked-example manifest + wiremock E2E (Bearer + developer-token together → rows). Depends on T2.

DAG: `T1 → T2 → T3` (chain — all touch overlapping files; serialized for clean review).

## 10 · Scope boundary — explicit
- **OUT (separate follow-up epics):**
  - **Read-vs-write gating** — let an idempotent POST "query" register as a *read* so a GAQL connection needn't set `allow_writes=true`.
  - **GAQL pagination** — `nextPageToken` loop on the action path (`send_request` is single-shot today).
  - **Per-table/per-action headers** (Approach ③) — only if a real need appears.

> **Not planned yet.** This notebook is the design artifact. Next: spec self-review → user review → `writing-plans` → `submit_plan`."